# Raw-ARI SMC vs Official Annealed TPS

This notebook compares complete runs of suffix direct-MH SMC and the official Algorithm 2 annealed TPS adapter. It never launches direct sampling, MBAR, mixture MCMC, or another SMC.

In [ ]:
# Mount Drive; clone/update this repository and the pinned official submodule.
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
from pathlib import Path
REPO_URL = 'https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main'
REPO_DIR = Path('/content/rea_of_llms_supplementary_code-main')
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--recurse-submodules', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'submodule', 'update', '--init', '--recursive'], check=True)
os.chdir(REPO_DIR)
print('repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('official TPS commit:', subprocess.check_output(['git', '-C', 'external/rea_of_llms_official', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# Install dependencies after every fresh Colab runtime.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Dependencies installed.')

In [ ]:
# Verify GPU availability before loading the model.
import torch
print('torch:', torch.__version__)
print('compiled CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# Fast correctness checks before expensive runs.
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'tests/test_suffix_direct_mh_smc.py',
                'tests/test_official_annealed_tps.py'], check=True)

In [ ]:
# EDIT PARAMETERS HERE
ALGORITHMS_TO_RUN = ['suffix_direct_mh_smc', 'official_annealed_tps']
SEEDS = [200, 201, 202]
MODEL_NAME = 'roneneldan/TinyStories-8M'
MAX_NEW_TOKENS = 100
ARI_MODE = 'raw'
LAMBDAS = ('0,-0.25,-0.5,-0.75,-1,-1.25,-1.5,-1.75,'
           '-2,-2.25,-2.5,-2.75,-3,-3.25,-3.5,-3.75,'
           '-4,-4.25,-4.5,-4.75,-5')
SMC_NUM_PARTICLES = 256
SMC_MCMC_STEPS_PER_LEVEL = 10
SMC_ESS_THRESHOLD = 0.5
SMC_SUFFIX_SIZES = '1,2,4,8,16,32,64,100'
SMC_SUFFIX_RHOS = '0.20,0.18,0.16,0.14,0.12,0.10,0.06,0.04'
TPS_STEPS_PER_BIAS = 2000
RUN_CONVERGENCE_STUDY = False
SMC_CONVERGENCE_GRID = [
    {'num_particles': 128, 'mcmc_steps_per_level': 5},
    {'num_particles': 256, 'mcmc_steps_per_level': 10},
    {'num_particles': 512, 'mcmc_steps_per_level': 10},
]
TPS_CONVERGENCE_GRID = [
    {'steps_per_bias': 500}, {'steps_per_bias': 1000},
    {'steps_per_bias': 2000}, {'steps_per_bias': 4000},
]
MEAN_ARI_ABS_TOLERANCE = 0.25
TAIL_PROB_ABS_TOLERANCE = 0.03
MIN_TPS_TEMPORAL_ESS = 100
MIN_SMC_FINAL_WEIGHT_ESS_OVER_N = 0.30
TAIL_THRESHOLDS = '10,12,15,18'
REFERENCE_SUMMARY_PATH = None
EXPERIMENT_LABEL = 'raw_ari_smc_vs_official_tps'
SKIP_COMPLETED_RUNS = True
DEVICE = 'cuda'

In [ ]:
# Validate all assumptions that make the comparison meaningful.
import math
assert set(ALGORITHMS_TO_RUN) <= {'suffix_direct_mh_smc', 'official_annealed_tps'}
assert len(SEEDS) == len(set(SEEDS))
assert ARI_MODE == 'raw', 'This benchmark requires uncapped raw ARI.'
lambda_values = [float(x) for x in LAMBDAS.split(',')]
assert lambda_values[0] == 0 and all(b <= a for a, b in zip(lambda_values, lambda_values[1:]))
if 'suffix_direct_mh_smc' in ALGORITHMS_TO_RUN:
    suffix_sizes = [int(x) for x in SMC_SUFFIX_SIZES.split(',')]
    suffix_rhos = [float(x) for x in SMC_SUFFIX_RHOS.split(',')]
    assert len(suffix_sizes) == len(suffix_rhos) and math.isclose(sum(suffix_rhos), 1.0)
    assert all(1 <= size <= MAX_NEW_TOKENS for size in suffix_sizes)
    assert MAX_NEW_TOKENS in suffix_sizes and suffix_rhos[suffix_sizes.index(MAX_NEW_TOKENS)] > 0
if DEVICE == 'cuda': assert torch.cuda.is_available(), 'Select a GPU Colab runtime.'
OUTPUT_ROOT = Path('/content/drive/MyDrive/rea_smc_vs_tps_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Operational convergence criteria:', {
    'mean_ari_abs_tolerance': MEAN_ARI_ABS_TOLERANCE,
    'tail_probability_abs_tolerance': TAIL_PROB_ABS_TOLERANCE,
    'min_tps_temporal_ess': MIN_TPS_TEMPORAL_ESS,
    'min_smc_final_weight_ess_over_n': MIN_SMC_FINAL_WEIGHT_ESS_OVER_N,
})

In [ ]:
# Build complete-run configurations. No run uses wall time as a stopping rule.
configurations = []
if not RUN_CONVERGENCE_STUDY:
    configurations.append({
        'label': 'selected', 'algorithms': ALGORITHMS_TO_RUN,
        'smc_num_particles': SMC_NUM_PARTICLES,
        'smc_mcmc_steps_per_level': SMC_MCMC_STEPS_PER_LEVEL,
        'tps_steps_per_bias': TPS_STEPS_PER_BIAS,
    })
else:
    if 'suffix_direct_mh_smc' in ALGORITHMS_TO_RUN:
        for row in SMC_CONVERGENCE_GRID:
            configurations.append({
                'label': f"smc_n{row['num_particles']}_m{row['mcmc_steps_per_level']}",
                'algorithms': ['suffix_direct_mh_smc'],
                'smc_num_particles': row['num_particles'],
                'smc_mcmc_steps_per_level': row['mcmc_steps_per_level'],
                'tps_steps_per_bias': TPS_STEPS_PER_BIAS,
            })
    if 'official_annealed_tps' in ALGORITHMS_TO_RUN:
        for row in TPS_CONVERGENCE_GRID:
            configurations.append({
                'label': f"tps_s{row['steps_per_bias']}",
                'algorithms': ['official_annealed_tps'],
                'smc_num_particles': SMC_NUM_PARTICLES,
                'smc_mcmc_steps_per_level': SMC_MCMC_STEPS_PER_LEVEL,
                'tps_steps_per_bias': row['steps_per_bias'],
            })
configurations

In [ ]:
# Execute each selected algorithm independently for every seed and full schedule.
from tqdm.auto import tqdm
for cfg in tqdm(configurations, desc='Complete configurations'):
    command = [
        sys.executable, 'smc_tps_comparison/run_comparison.py',
        '--algorithms', ','.join(cfg['algorithms']), '--seeds', ','.join(map(str, SEEDS)),
        '--model-name', MODEL_NAME, '--device', DEVICE,
        '--max-new-tokens', str(MAX_NEW_TOKENS), '--ari-mode', ARI_MODE,
        '--lambdas', LAMBDAS, '--tail-thresholds', TAIL_THRESHOLDS,
        '--smc-num-particles', str(cfg['smc_num_particles']),
        '--smc-mcmc-steps-per-level', str(cfg['smc_mcmc_steps_per_level']),
        '--smc-ess-threshold', str(SMC_ESS_THRESHOLD),
        '--smc-suffix-sizes', SMC_SUFFIX_SIZES, '--smc-suffix-rhos', SMC_SUFFIX_RHOS,
        '--tps-steps-per-bias', str(cfg['tps_steps_per_bias']),
        '--output-root', str(OUTPUT_ROOT), '--experiment-label', EXPERIMENT_LABEL,
        '--configuration-label', cfg['label'],
    ]
    if SKIP_COMPLETED_RUNS: command.append('--skip-completed')
    if REFERENCE_SUMMARY_PATH is not None:
        command += ['--reference-summary-path', str(REFERENCE_SUMMARY_PATH)]
    subprocess.run(command, check=True)

In [ ]:
# Load and validate every completed output and fingerprinted run.
import json, numpy as np, pandas as pd
comparison_payloads, run_summaries = {}, []
for cfg in configurations:
    root = OUTPUT_ROOT / EXPERIMENT_LABEL / cfg['label']
    payload = json.loads((root / 'comparison_summary.json').read_text())
    assert payload['ari_mode'] == 'raw'
    comparison_payloads[cfg['label']] = payload
    for row in payload['runs']:
        assert row['completed_full_schedule']
        summary_path = root / row['algorithm'] / f"seed_{row['seed']}" / 'summary.json'
        summary = json.loads(summary_path.read_text())
        assert summary['configuration_fingerprint'] == row['configuration_fingerprint']
        assert summary['ari_mode'] == 'raw'
        if row['algorithm'] == 'suffix_direct_mh_smc':
            assert summary['mutation_kernel'] == 'suffix_direct_mh' and summary['resample_final'] is True
        else:
            assert summary['resampling'] is False and summary['importance_weights'] is False
        run_summaries.append(summary | {'configuration': cfg['label']})
runs_df = pd.DataFrame([row for payload in comparison_payloads.values() for row in payload['runs']])
display(runs_df[['algorithm','configuration','seed','completed_full_schedule','runtime_seconds',
                 'total_proposals','total_regenerated_tokens','acceptance_rate',
                 'mean_raw_ari_final_target','max_raw_ari','tail_15','tail_18',
                 'smc_weight_ess_over_n','tps_temporal_ess']])

In [ ]:
# Aggregate independent seeds: mean, standard deviation, CI, and empirical efficiency.
aggregate_df = pd.DataFrame([
    row | {'configuration': label}
    for label, payload in comparison_payloads.items()
    for row in payload['aggregate_estimators']
])
display(aggregate_df)
if REFERENCE_SUMMARY_PATH is None:
    print('No independent ground truth: MSE/RMSE is not computed.')

In [ ]:
# Operational convergence criteria: compare each budget with the next larger complete budget.
operational_rows = []
if RUN_CONVERGENCE_STUDY:
    for algorithm in sorted(runs_df['algorithm'].unique()):
        subset = runs_df[runs_df['algorithm'] == algorithm]
        budgets = subset.groupby('configuration')['runtime_seconds'].mean().sort_values().index.tolist()
        for current_label, next_label in zip(budgets, budgets[1:]):
            current = subset[subset['configuration'] == current_label]
            higher = subset[subset['configuration'] == next_label]
            stable = (abs(current['mean_raw_ari_final_target'].mean() - higher['mean_raw_ari_final_target'].mean()) <= MEAN_ARI_ABS_TOLERANCE and
                      abs(current['tail_15'].mean() - higher['tail_15'].mean()) <= TAIL_PROB_ABS_TOLERANCE and
                      abs(current['tail_18'].mean() - higher['tail_18'].mean()) <= TAIL_PROB_ABS_TOLERANCE)
            diagnostic_ok = (current['smc_weight_ess_over_n'].min() >= MIN_SMC_FINAL_WEIGHT_ESS_OVER_N
                             if algorithm == 'suffix_direct_mh_smc'
                             else current['tps_temporal_ess'].min() >= MIN_TPS_TEMPORAL_ESS)
            operational_rows.append({'algorithm': algorithm, 'configuration': current_label,
                                     'compared_with': next_label, 'stable_estimators': stable,
                                     'ess_criterion': diagnostic_ok,
                                     'meets_operational_criteria': stable and diagnostic_ok})
    operational_df = pd.DataFrame(operational_rows)
    display(operational_df)
    selected_operational = operational_df[operational_df['meets_operational_criteria']].groupby('algorithm').first().reset_index()
    display(selected_operational)
    selected_runs = runs_df.merge(selected_operational[['algorithm','configuration']], on=['algorithm','configuration'])
    final_operational_comparison = selected_runs.groupby(['algorithm','configuration']).agg(
        mean_runtime_seconds=('runtime_seconds','mean'),
        mean_raw_ari=('mean_raw_ari_final_target','mean'),
        std_raw_ari=('mean_raw_ari_final_target','std'),
        mean_tail_15=('tail_15','mean'), mean_tail_18=('tail_18','mean'),
        min_smc_weight_ess_over_n=('smc_weight_ess_over_n','min'),
        min_tps_temporal_ess=('tps_temporal_ess','min')).reset_index()
    display(final_operational_comparison)
else:
    print('Convergence grid disabled: operational convergence is not assessed from one budget.')

In [ ]:
# Estimator stability, runtime, tail probabilities, and discovery maximum.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
metrics = ['mean_raw_ari_final_target','tail_15','tail_18','max_raw_ari']
plot_summary = runs_df.groupby(['algorithm','configuration']).agg(
    runtime_seconds=('runtime_seconds','mean'),
    **{f'{metric}_mean': (metric,'mean') for metric in metrics},
    **{f'{metric}_std': (metric,'std') for metric in metrics}).reset_index()
for algorithm, group in plot_summary.groupby('algorithm'):
    group = group.sort_values('runtime_seconds')
    for ax, metric in zip(axes.flat, metrics):
        ax.errorbar(group['runtime_seconds'], group[f'{metric}_mean'],
                    yerr=group[f'{metric}_std'].fillna(0), marker='o', capsize=3, label=algorithm)
axes[0,0].set(title='Mean-ARI stability over complete-run budget', xlabel='mean complete-run runtime (s)', ylabel='estimate')
axes[0,1].set(title='P(ARI >= 15) stability over complete-run budget', xlabel='mean complete-run runtime (s)', ylabel='estimate')
axes[1,0].set(title='P(ARI >= 18) stability over complete-run budget', xlabel='mean complete-run runtime (s)', ylabel='estimate')
axes[1,1].set(title='Tail discovery maximum (not target accuracy)', xlabel='mean complete-run runtime (s)', ylabel='max raw ARI')
for ax in axes.flat: ax.grid(alpha=.25); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Final-target ECDF: final-resampled particles for SMC, temporal samples for TPS.
plt.figure(figsize=(9, 6))
for summary in run_summaries:
    root = OUTPUT_ROOT / EXPERIMENT_LABEL / summary['configuration'] / summary['algorithm'] / f"seed_{summary['seed']}"
    if summary['algorithm'] == 'suffix_direct_mh_smc':
        frame = pd.read_json(root / 'samples.jsonl', lines=True).sort_values('raw_ari')
        x, y = frame['raw_ari'].to_numpy(), frame['normalized_weight'].cumsum().to_numpy()
    else:
        with np.load(root / 'trajectory.npz') as trajectory:
            final_index = trajectory['lambda_index'].max()
            x = np.sort(trajectory['raw_ari'][trajectory['lambda_index'] == final_index])
            y = np.arange(1, len(x)+1) / len(x)
    plt.step(x, y, where='post', alpha=.7, label=f"{summary['algorithm']} seed {summary['seed']}")
plt.xlabel('raw ARI'); plt.ylabel('cumulative target mass / temporal fraction')
plt.title('Final-target ECDF'); plt.grid(alpha=.25); plt.legend(fontsize=7); plt.show()

In [ ]:
# Highest distinct completions observed by each algorithm.
extreme_rows = []
for summary in run_summaries:
    for row in summary.get('top_discovered_completions', []):
        extreme_rows.append({'algorithm': summary['algorithm'], 'seed': summary['seed'],
                             'raw_ari': row['raw_ari'], 'lambda': row['lambda'],
                             'time_first_seen': row['time_first_seen_seconds'],
                             'completion': row['completion']})
extreme_df = (pd.DataFrame(extreme_rows).sort_values('raw_ari', ascending=False)
              .drop_duplicates(['algorithm','completion']).groupby('algorithm').head(15))
pd.set_option('display.max_colwidth', None)
display(extreme_df)

In [ ]:
# Additional mixing, genealogy, weight, and token-repetition diagnostics.
# This cell reloads completed runs from disk and safely skips incomplete outputs.
import json
import numpy as np
import pandas as pd
from rea_llms.benchmarking import temporal_ess_initial_positive

def consecutive_token_repeats(token_ids):
    ids = np.asarray(list(token_ids), dtype=np.int64)
    return int(np.sum(ids[:-1] == ids[1:])) if ids.size > 1 else 0

selected_configurations = globals().get('configurations') or [
    {'label': 'selected', 'algorithms': ALGORITHMS_TO_RUN}
]
diagnostic_summaries = []
missing_outputs = []
for cfg in selected_configurations:
    for algorithm in cfg['algorithms']:
        for seed in SEEDS:
            run_root = (OUTPUT_ROOT / EXPERIMENT_LABEL / cfg['label'] /
                        algorithm / f'seed_{seed}')
            summary_path = run_root / 'summary.json'
            if not summary_path.exists():
                missing_outputs.append(str(summary_path))
                continue
            summary = json.loads(summary_path.read_text(encoding='utf-8'))
            if not summary.get('completed_full_schedule', False):
                missing_outputs.append(f'{summary_path} (incomplete schedule)')
                continue
            diagnostic_summaries.append((summary, cfg['label'], run_root))

if missing_outputs:
    print('Skipped missing or incomplete outputs:')
    for path in missing_outputs:
        print(' -', path)

diagnostic_rows = []
for summary, configuration, run_root in diagnostic_summaries:
    algorithm = summary['algorithm']
    row = {
        'algorithm': algorithm,
        'configuration': configuration,
        'seed': int(summary['seed']),
        'runtime_seconds': float(summary['runtime_total_seconds']),
        'mean_final_ari': float(summary['mean_raw_ari_final_target']),
    }

    if algorithm == 'suffix_direct_mh_smc':
        samples_path = run_root / 'samples.jsonl'
        if not samples_path.exists():
            print('Skipping SMC diagnostics without samples:', samples_path)
            continue
        samples = pd.read_json(samples_path, lines=True)
        required = {'completion_token_ids', 'normalized_weight'}
        if not required.issubset(samples.columns):
            raise ValueError(f'{samples_path} is missing columns {required - set(samples.columns)}')
        repeats = samples['completion_token_ids'].map(consecutive_token_repeats).to_numpy(float)
        weights = pd.to_numeric(samples['normalized_weight']).to_numpy(float)
        if not np.all(np.isfinite(weights)) or weights.sum() <= 0:
            raise ValueError(f'Invalid normalized weights in {samples_path}')
        weights = weights / weights.sum()
        num_particles = int(summary.get('num_particles', len(samples)))
        distinct_ancestors = int(summary.get(
            'final_distinct_root_ancestors', samples['ancestor_id'].nunique()))
        row.update({
            'weight_ess': float(summary.get('smc_weight_ess', 1.0 / np.sum(weights**2))),
            'unique_completion_fraction': float(summary.get(
                'unique_completion_fraction', samples['completion'].nunique() / len(samples))),
            'distinct_root_ancestors': distinct_ancestors,
            'root_ancestor_fraction': distinct_ancestors / num_particles,
            'resampling_steps': int(summary['resampling_steps']),
            'largest_weight': float(weights.max()),
            'weighted_mean_token_repeats': float(np.sum(weights * repeats)),
        })
    elif algorithm == 'official_annealed_tps':
        trajectory_path = run_root / 'trajectory.npz'
        if not trajectory_path.exists():
            print('Skipping TPS diagnostics without trajectory:', trajectory_path)
            continue
        with np.load(trajectory_path) as trajectory:
            lambda_index = np.asarray(trajectory['lambda_index'])
            raw_ari = np.asarray(trajectory['raw_ari'], dtype=float)
        if lambda_index.size == 0 or raw_ari.size != lambda_index.size:
            raise ValueError(f'Invalid TPS trajectory in {trajectory_path}')
        final_ari = raw_ari[lambda_index == lambda_index.max()]
        if final_ari.size == 0 or not np.all(np.isfinite(final_ari)):
            raise ValueError(f'Invalid final TPS ARI values in {trajectory_path}')
        halfway = final_ari.size // 2
        first_mean = float(final_ari[:halfway].mean()) if halfway else np.nan
        second_mean = float(final_ari[halfway:].mean())
        row.update({
            'temporal_ess': float(temporal_ess_initial_positive(final_ari)),
            'first_half_mean': first_mean,
            'second_half_mean': second_mean,
            'within_level_drift': abs(second_mean - first_mean) if np.isfinite(first_mean) else np.nan,
        })
    else:
        continue
    diagnostic_rows.append(row)

if not diagnostic_rows:
    raise FileNotFoundError('No complete SMC or TPS outputs were found for the selected configuration.')

diagnostics_df = (pd.DataFrame(diagnostic_rows)
                  .sort_values(['algorithm', 'configuration', 'seed'])
                  .reset_index(drop=True))
display(diagnostics_df.set_index(['algorithm', 'configuration', 'seed']).T)